In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
"""
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
"""
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

'\nimport numpy as np # linear algebra\nimport pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)\n\n# Input data files are available in the read-only "../input/" directory\n# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory\n\nimport os\nfor dirname, _, filenames in os.walk(\'/kaggle/input\'):\n    for filename in filenames:\n        print(os.path.join(dirname, filename))\n'

In [2]:
import os
import glob
import random
import numpy as np
import cv2
from PIL import Image
from transformers import pipeline

2025-04-12 18:19:56.370488: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744481996.637988      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744481996.708142      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
train_dir = "/kaggle/input/ethz-cil-monocular-depth-estimation-2025/train/train"
test_dir = "/kaggle/input/ethz-cil-monocular-depth-estimation-2025/test/test"
output_width, output_height = 560, 426
# Two output directories: one for .npy files and one for visual images.
output_npy_dir = "/kaggle/working/da2_pipe_preds_npy"
output_img_dir = "/kaggle/working/da2_pipe_preds_img"
os.makedirs(output_npy_dir, exist_ok=True)
os.makedirs(output_img_dir, exist_ok=True)

# Load test list
with open("/kaggle/input/ethz-cil-monocular-depth-estimation-2025/test_list.txt", "r") as f:
    test_list = [line.strip().split()[0] for line in f]

In [4]:
pipe = pipeline(task="depth-estimation", model="depth-anything/Depth-Anything-V2-Large-hf", device=0)

config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


In [5]:
all_rgb_paths = glob.glob(os.path.join(train_dir, "*_rgb.png"))
base_names = [os.path.basename(path).replace("_rgb.png", "") for path in all_rgb_paths]

# Randomly sample a subset for calibration (e.g. 1000 samples; adjust as needed)
num_samples = 1000
sample_names = random.sample(base_names, min(num_samples, len(base_names)))

all_preds = []  # to store flattened predicted depths
all_gts   = []  # to store flattened ground truth depths

for name in sample_names:
    rgb_path = os.path.join(train_dir, f"{name}_rgb.png")
    depth_path = os.path.join(train_dir, f"{name}_depth.npy")
    
    # Load the RGB image using PIL (pipeline expects PIL images)
    image = Image.open(rgb_path).convert("RGB")
    # Run the pipeline: the result is a dictionary with key "depth"
    result = pipe(image)
    # Access the depth map; if the pipeline returns a dict directly:
    depth_pred = result["depth"]  # depth_pred is assumed to be a NumPy array or can be converted to one
    
    # Convert to NumPy array in case it's not already
    depth_pred = np.array(depth_pred)
    # Resize prediction to the original resolution (width, height) if needed
    depth_pred_resized = cv2.resize(depth_pred, (output_width, output_height))
    
    # Load ground truth depth (assumed to be in meters)
    gt_depth = np.load(depth_path)
    
    # Flatten and collect for calibration
    all_preds.append(depth_pred_resized.flatten())
    all_gts.append(gt_depth.flatten())

# Concatenate all predictions and ground truths
preds_flat = np.concatenate(all_preds)
gts_flat = np.concatenate(all_gts)

# Perform a least-squares fit to compute linear calibration parameters:
# We want to find a and b such that: gt ≈ a * pred + b
A = np.vstack([preds_flat, np.ones_like(preds_flat)]).T
a, b = np.linalg.lstsq(A, gts_flat, rcond=None)[0]
print(f"Calibrated scale (a): {a}")
print(f"Calibrated shift (b): {b}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Calibrated scale (a): -0.015217554954262719
Calibrated shift (b): 4.275360951082471


In [6]:
with open("/kaggle/input/ethz-cil-monocular-depth-estimation-2025/test_list.txt", "r") as f:
    test_list = [line.strip().split()[0] for line in f]

for filename in test_list:
    # Load the test image as a PIL image
    img_path = os.path.join(test_dir, filename)
    image = Image.open(img_path).convert("RGB")
    
    # Get the raw depth prediction using the pipeline
    result = pipe(image)
    depth_raw = np.array(result["depth"])
    
    # Apply calibration: convert raw prediction into metric depth
    depth_calibrated = a * depth_raw + b
    # Clip negative (or near zero) values
    depth_calibrated = np.maximum(depth_calibrated, 0.1)
    
    # Resize to the required output resolution
    depth_final = cv2.resize(depth_calibrated, (output_width, output_height))
    
    # Save as .npy file
    base = filename.replace("_rgb.png", "_depth.npy")
    save_path_npy = os.path.join(output_npy_dir, base)
    np.save(save_path_npy, depth_final)
    
    # Create a visualization by normalizing and applying a colormap
    depth_vis = cv2.normalize(depth_final, None, 0, 255, cv2.NORM_MINMAX)
    depth_vis = np.uint8(depth_vis)
    depth_vis = cv2.applyColorMap(depth_vis, cv2.COLORMAP_JET)
    save_path_img = os.path.join(output_img_dir, base.replace(".npy", ".png"))
    cv2.imwrite(save_path_img, depth_vis)